# Snap-to-moment start model (Le Xin)

**Idea.** The timing model guesses "move the start by X ms". But a sung note really begins at a *moment you can
find in the audio*: a loudness attack, a jump in loudness, the voice switching on, or the pitch changing.
So instead of guessing a number, we list the real moments near each note's start (plus "stay where you are")
and train a model to **pick the right one**, like a multiple-choice question instead of a fill-in-the-blank.

**Why it is worth it (ceiling check, training songs).** On the final chain (timing → split → delete, onset F1
0.3667), always picking the best nearby moment would give 0.5137. For 92.5% of the checkable notes a moment
(or staying) lies within 50 ms of the true start, while today only 65.8% of those starts are within 50 ms.

**Where it sits.** A 4th step after the combined chain: it only moves note **starts**; ends, pitches, splits
and deletions are kept. Input = `exports/combined_final_train_oof_v1.csv` (train, out-of-fold by song).

**Versions.** Steps 1-3 = picker v1 (20 clues, exports `le_xin_snap_*_v1`). Step 4 adds better clues
(Basic Pitch activations, pitch settle, neighbours, comparison with the other options) and retrains on the
training songs; Step 5 checks picker v2 once on val and exports `le_xin_snap_*_v2`. The v1 exports stay as they are.

**Rules kept.** Clues come from the audio and the predicted notes only. The answer key (`references_train.csv`)
is used only to make the labels and to score. Test data is never opened. TextGrid is not used.

In [1]:
# Setup: paths, the input note table (final combined chain), answer key for labels/scoring only
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

HERE = next(p / 'notebooks' / 'correction_handover' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'selected_100.csv').exists())
ROOT = (HERE / '..' / '..').resolve()
sys.path.insert(0, str(HERE))
import correction_utils as cu
import librosa
DATA, EXPORTS, CACHE_DIR = ROOT / 'basic_pitch_runs' / 'correction_handover', HERE / 'exports', HERE / 'le_xin_cache'
read = lambda path: pd.read_csv(path, dtype={'note_id':str, 'recording_id':str, 'sample_id':str})

manifest   = pd.read_csv(DATA / 'training_recording_manifest.csv', dtype={'recording_id':str, 'sample_id':str})
references = read(DATA / 'references_train.csv')                  # answer key: labels and scoring only
folds      = pd.read_csv(DATA / 'folds.csv').set_index('sample_id')['fold']
assert manifest['split'].eq('train').all()                         # test stays locked
chain = read(EXPORTS / 'combined_final_train_oof_v1.csv')         # timing -> split -> delete, out-of-fold by song
cu.validate_notes(chain, manifest)
print(f'Input: {len(chain)} notes in {chain.recording_id.nunique()} recordings (final combined chain, train out-of-fold)')

Input: 7550 notes in 534 recordings (final combined chain, train out-of-fold)


In [2]:
# Step 1: list the real moments near every note's start, describe each one with clues, and label the right one
SR, HOP = 22050, 256           # ~11.6 ms frames, same as the timing model
WIN = 0.10                     # look for moments within +-100 ms of the current start
FRAME = HOP / SR
FAR = 12.0                     # 'no pitch found' stand-in: one octave
F0 = np.load(HERE / 'hann_qian_pitch_model' / 'f0_tracks.npz')     # Hann Qian's cached pyin tracks (train + val only)

def audio_curves(rec):
    """Per-frame curves for one recording: attack strength, loudness change, sung pitch (MIDI, NaN = silent)."""
    y, _ = librosa.load(ROOT / rec.audio_path, sr=SR, mono=True)
    flux = librosa.onset.onset_strength(y=y, sr=SR, hop_length=HOP)
    flux = flux / (np.percentile(flux, 99) + 1e-9)                               # scale per recording
    db = librosa.amplitude_to_db(librosa.feature.rms(y=y, hop_length=HOP)[0], ref=np.max)
    rise = np.diff(db, prepend=db[0])                                            # dB change per frame
    t_pitch, f0 = F0[f'{rec.recording_id}_t'], F0[f'{rec.recording_id}_f0']
    n = min(len(flux), len(rise), len(f0))
    midi = librosa.hz_to_midi(f0[:n])
    voiced = ~np.isnan(midi)
    jump = np.abs(np.diff(np.nan_to_num(midi), prepend=0.0))
    jump[~voiced] = 0; jump[1:][~voiced[:-1]] = 0                                # ignore jumps into/out of silence
    return dict(t=librosa.frames_to_time(np.arange(n), sr=SR, hop_length=HOP),
                flux=flux[:n], rise=rise[:n], midi=midi, voiced=voiced, jump=jump)

def peaks(v, height):
    return np.flatnonzero((v[1:-1] > v[:-2]) & (v[1:-1] >= v[2:]) & (v[1:-1] >= height)) + 1

def moment_frames(c):
    """Every frame that is a 'moment', with the kinds of evidence behind it."""
    kinds = {'attack': peaks(c['flux'], 0.1),                                    # loudness attack peak
             'louder': peaks(c['rise'], 1.0),                                    # loudness rises >= 1 dB in a frame
             'voice_on': np.flatnonzero(c['voiced'][1:] & ~c['voiced'][:-1]) + 1, # silence -> singing
             'pitch_jump': np.flatnonzero(c['jump'] >= 0.5)}                     # pitch moves >= half a semitone
    return kinds

def sung_gap(c, a, b, pitch):
    """How far (semitones) the sung pitch between times a and b is from the note's pitch."""
    seg = c['midi'][(c['t'] >= a) & (c['t'] < b)]
    seg = seg[~np.isnan(seg)]
    return min(abs(np.median(seg) - pitch), FAR) if len(seg) else FAR

def voiced_share(c, a, b):
    m = (c['t'] >= a) & (c['t'] < b)
    return float(c['voiced'][m].mean()) if m.any() else 0.0

def candidates_for_recording(rec, notes):
    c = audio_curves(rec)
    kinds = moment_frames(c)
    all_frames = np.unique(np.concatenate(list(kinds.values())))
    near_kind = {k: set(np.concatenate([f - 1, f, f + 1])) for k, f in kinds.items()}   # +-1 frame counts as the same moment
    rows = []
    for n in notes.itertuples():
        here = min(int(round(n.start / FRAME)), len(c['t']) - 1)                  # frame of the current start
        frames = all_frames[(np.abs(c['t'][all_frames] - n.start) <= WIN) & (np.abs(all_frames - here) > 2)]
        # merge moments closer than 2 frames (keep the first): they are the same event; moments within
        # 2 frames of the current start are the same as 'stay' (their kinds are credited to 'stay')
        merged = [f for i, f in enumerate(frames) if i == 0 or f - frames[i - 1] > 2]
        options = [(n.start, here, True)] + [(c['t'][f], f, False) for f in merged]   # 'stay' is always option 0
        for t, i, stay in options:
            lo, hi = max(i - 1, 0), min(i + 2, len(c['t']))
            near = set(range(i - 2, i + 3)) if stay else {i}
            rows.append(dict(note_id=n.note_id, recording_id=n.recording_id, t=t, dt=t - n.start,
                             is_stay=float(stay),
                             **{f'is_{k}': float(bool(near & near_kind[k])) for k in kinds},
                             attack=c['flux'][lo:hi].max(), louder=c['rise'][lo:hi].max(), pitch_jump=c['jump'][lo:hi].max(),
                             voiced_before=voiced_share(c, t - 0.05, t), voiced_after=voiced_share(c, t, t + 0.05),
                             gap_before=sung_gap(c, t - 0.05, t, n.pitch), gap_after=sung_gap(c, t, t + 0.05, n.pitch),
                             note_dur=n.end - n.start, note_pitch=n.pitch, is_piece=float('~' in n.note_id),
                             prev_gap=n.prev_gap))
    return rows

def build_candidates(notes, man):
    """Candidate moments + clues for every note (audio and predicted notes only; no answer key)."""
    notes = notes.sort_values(['recording_id', 'start']).copy()
    notes['prev_gap'] = (notes.start - notes.groupby('recording_id').end.shift(1)).fillna(0.0)
    rows = []
    for rec in man.itertuples():
        assert rec.audio_path.startswith(('data/train/', 'data/val/'))           # test audio stays locked
        rows += candidates_for_recording(rec, notes[notes.recording_id == rec.recording_id])
    cands = pd.DataFrame(rows)
    g = cands.groupby('note_id')
    cands['n_options'] = g.t.transform('size')
    cands['attack_rank'] = g.attack.rank(ascending=False, method='min')          # 1 = strongest attack for this note
    cands['abs_dt'] = cands.dt.abs()
    return cands

# Labels (answer key, never a clue): the note's true start = nearest reference start with the same pitch
# (within 50 cents) within +-150 ms. An option is RIGHT if it lies within 50 ms of it (what the scorer counts as a hit).
REF = {rid: g for rid, g in references.groupby('recording_id')}
def true_start(n):
    g = REF.get(n.recording_id)
    if g is None:
        return np.nan
    g = g[(np.abs(g.pitch - n.pitch) < 0.5) & (np.abs(g.start - n.start) <= 0.15)]
    return g.start.iloc[np.argmin(np.abs(g.start - n.start))] if len(g) else np.nan

def add_labels(cands, notes):
    target = pd.Series({n.note_id: true_start(n) for n in notes.itertuples()})
    out = cands.copy()
    out['_target'] = out.note_id.map(target)
    out['label'] = ((out.t - out._target).abs() <= 0.05).astype(int)
    out['labelled_note'] = out._target.notna()                                       # False = no usable true start
    return out.drop(columns='_target')

CACHE = CACHE_DIR / 'snap_candidates_train_v1.csv'
if CACHE.exists():
    cands = read(CACHE)
    print('Loaded cached candidates:', CACHE.name)
else:
    cands = add_labels(build_candidates(chain, manifest), chain)              # ~1-2 minutes
    cands.to_csv(CACHE, index=False)
    print('Built and cached candidates:', CACHE.name)
cands['sample_id'] = cands.recording_id.map(manifest.set_index('recording_id').sample_id)
cands['fold'] = cands.sample_id.map(folds)
assert cands.fold.notna().all() and set(cands.note_id) == set(chain.note_id)
CLUES = ['dt', 'abs_dt', 'is_stay', 'is_attack', 'is_louder', 'is_voice_on', 'is_pitch_jump', 'attack', 'louder',
         'pitch_jump', 'voiced_before', 'voiced_after', 'gap_before', 'gap_after', 'note_dur', 'note_pitch',
         'is_piece', 'prev_gap', 'n_options', 'attack_rank']

# What the table looks like
per_note = cands.groupby('note_id').agg(options=('t', 'size'), labelled=('labelled_note', 'first'), has_right=('label', 'max'))
lab = per_note[per_note.labelled]
right = cands[cands.label == 1]
summary = pd.Series({
    'notes': len(per_note),
    'candidate rows (options)': len(cands),
    'options per note (median)': per_note.options.median(),
    'notes with a usable true start': len(lab),
    '... of which an option is within 50 ms (the model CAN be right)': f"{lab.has_right.mean():.1%}",
    '... of which "stay" is already right': f"{cands[cands.is_stay == 1].groupby('note_id').label.max().reindex(lab.index).mean():.1%}",
    '... of which only a MOVE is right (where the model can gain)': f"{(lab.has_right & ~cands[cands.is_stay == 1].set_index('note_id').label.reindex(lab.index).astype(bool)).mean():.1%}",
    'share of options that are right': f"{cands.label.mean():.1%}",
})
display(summary.to_frame('value'))
kinds = ['is_stay', 'is_attack', 'is_louder', 'is_voice_on', 'is_pitch_jump']
display(pd.DataFrame({'options': cands[kinds].sum(), 'right options': right[kinds].sum(),
                      'share right': (right[kinds].sum() / cands[kinds].sum()).round(3)}).rename_axis('kind of moment'))
print('Right MOVE options, how far they move the start (ms):', (right[right.is_stay == 0].dt * 1000).describe()[['25%', '50%', '75%']].round(0).to_dict())
display(cands[['note_id', 't', 'dt'] + kinds + ['attack', 'gap_after', 'label']].head(8))

Loaded cached candidates: snap_candidates_train_v1.csv


,value
notes,7550
candidate rows (options),22615
options per note (median),3.0
notes with a usable true start,4285
... of which an option is within 50 ms (the model CAN be right),91.1%
"... of which ""stay"" is already right",65.8%
... of which only a MOVE is right (where the model can gain),25.3%
share of options that are right,27.7%


,options,right options,share right
kind of moment,,,
is_stay,7550.0,2818.0,0.373
is_attack,17798.0,4725.0,0.265
is_louder,5827.0,1874.0,0.322
is_voice_on,3427.0,962.0,0.281
is_pitch_jump,6280.0,2377.0,0.379


Right MOVE options, how far they move the start (ms): {'25%': -52.0, '50%': 32.0, '75%': 48.0}


,note_id,t,dt,is_stay,is_attack,is_louder,is_voice_on,is_pitch_jump,attack,gap_after,label
0,c2da076a65ceaf6a699a:n00000,0.332033,0.000000,1.0,1.0,0.0,0.0,1.0,0.145784,4.40,1
1,c2da076a65ceaf6a699a:n00000,0.371519,0.039486,0.0,1.0,0.0,0.0,0.0,1.405158,0.10,0
2,c2da076a65ceaf6a699a:n00001,0.679697,0.000000,1.0,1.0,0.0,0.0,0.0,0.227168,5.20,1
3,c2da076a65ceaf6a699a:n00001,0.719819,0.040122,0.0,1.0,1.0,1.0,0.0,0.942590,1.35,0
4,c2da076a65ceaf6a699a:n00002,1.021678,0.000000,1.0,1.0,0.0,1.0,1.0,0.139181,0.00,1
5,c2da076a65ceaf6a699a:n00002,0.986848,-0.034830,0.0,1.0,0.0,0.0,0.0,0.134791,4.05,1
6,c2da076a65ceaf6a699a:n00002,1.068118,0.046440,0.0,1.0,0.0,0.0,0.0,1.009560,0.40,0
7,c2da076a65ceaf6a699a:n00003,1.321504,0.000000,1.0,1.0,1.0,1.0,1.0,0.171600,0.10,0


In [3]:
# Step 2: train the picker out-of-fold by song (rule vs decision tree vs random forest), choose its caution fairly, score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

ALL = sorted(cands.fold.unique())
train_rows = cands.labelled_note.to_numpy()          # only notes with a usable true start can teach the model
y = cands.label.to_numpy()
rec_fold = manifest.set_index('recording_id').sample_id.map(folds)
MARGINS = [-0.30, -0.20, -0.15, -0.10, -0.05, 0.0, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50, 9.0]   # < 0: move even if 'stay' is slightly likelier; 9.0 = never move

def oof_prob(make_model, fold_set=ALL, clues=None, table=None):
    """P(option is right) for every option, from a model that never saw that option's song."""
    clues, table = clues or CLUES, cands if table is None else table   # table: same rows as cands, more clue columns
    p = np.full(len(table), np.nan)
    for k in fold_set:
        test = (table.fold == k).to_numpy()
        train = table.fold.isin([f for f in fold_set if f != k]).to_numpy() & train_rows
        p[test] = make_model().fit(table.loc[train, clues], y[train]).predict_proba(table.loc[test, clues])[:, 1]
    return p

def choose(prob, margin, table=None):
    """Per note: move to the best MOVE option only if it beats 'stay' by at least `margin`; else stay."""
    c = (cands if table is None else table)[['note_id', 't', 'is_stay']].assign(p=prob)
    stay_p = c[c.is_stay == 1].set_index('note_id').p
    moves = c[c.is_stay == 0].dropna(subset=['p'])
    best = moves.loc[moves.groupby('note_id').p.idxmax()].set_index('note_id')
    go = best.p - stay_p.reindex(best.index) >= margin
    return best.loc[go, 't']                                        # note_id -> new start

def apply_starts(new_start, table=None):
    """COMPLETE note table: chosen notes get their new start; ends, pitches, IDs unchanged."""
    out = (chain if table is None else table)[cu.NOTE_COLUMNS].copy()
    s = out.note_id.map(new_start)
    out['start'] = np.where(s.notna(), s, out.start)
    out['start'] = np.minimum(out.start, out.end - 0.01)            # keep every note at least 10 ms long
    return out

def f1_on(fold_set, new_start):
    keep = manifest[manifest.recording_id.map(rec_fold).isin(fold_set)]
    t = apply_starts(new_start)
    t = t[t.recording_id.isin(keep.recording_id)]
    s = cu.evaluate_end_to_end(t, references[references.recording_id.isin(keep.recording_id)], keep)['summary']
    return s['song_macro_f1'], s['song_macro_offset_f1']

def nested(make_model, clues=None, table=None):
    """Out-of-fold probabilities + a margin chosen per held-out fold on the other 4 folds only."""
    outer = oof_prob(make_model, ALL, clues, table)
    picked, starts = {}, []
    for k in ALL:
        others = [f for f in ALL if f != k]
        inner = oof_prob(make_model, others, clues, table)
        m = max(MARGINS, key=lambda g: f1_on(others, choose(inner, g))[0])
        picked[int(k)] = m
        ch = choose(outer, m)
        starts.append(ch[ch.index.map(cands.drop_duplicates('note_id').set_index('note_id').fold) == k])
    return outer, pd.concat(starts), picked

# 1. Rule (no learning): if the sung pitch just after the current start does not match the note (>= 1 semitone off),
#    move to the nearest option where it does match (< 0.5 semitone).
stay_bad = cands[(cands.is_stay == 1) & (cands.gap_after >= 1.0)].note_id
ok = cands[(cands.is_stay == 0) & (cands.gap_after < 0.5) & cands.note_id.isin(stay_bad)]
rule_start = ok.loc[ok.groupby('note_id').abs_dt.idxmin()].set_index('note_id').t

# 2. Learned pickers
make_dt = lambda: DecisionTreeClassifier(min_samples_leaf=40, class_weight='balanced', random_state=0)
make_rf = lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=5, max_features=0.5,
                                         class_weight='balanced_subsample', n_jobs=-1, random_state=0)
dt_prob, dt_start, dt_margin = nested(make_dt)                      # ~1 minute
rf_prob, rf_start, rf_margin = nested(make_rf)                      # ~3-5 minutes
print('Margins chosen per fold (decision tree):', dt_margin)
print('Margins chosen per fold (random forest):', rf_margin)

# 3. Score: end-to-end on every training recording + how many labelled notes now start within 50 ms
truth = cands[cands.is_stay == 1].set_index('note_id').labelled_note
def note_hits(new_start):
    t = apply_starts(new_start).set_index('note_id')
    right = cands[cands.label == 1].groupby('note_id').t.apply(np.array)
    lab_ids = truth[truth].index
    hit = [np.any(np.abs(right.get(n, np.array([])) - t.at[n, 'start']) <= 1e-6) for n in lab_ids]
    return np.mean(hit)

rows = []
for name, starts in [('Final chain (no snap)', pd.Series(dtype=float)), ('Rule: snap to where pitch matches', rule_start),
                     ('Decision tree picker', dt_start), ('Random forest picker', rf_start)]:
    s = cu.evaluate_end_to_end(apply_starts(starts), references, manifest)['summary']
    assert s['recordings'] == len(manifest)
    rows.append(dict(version=name, starts_moved=len(starts), labelled_notes_right=f'{note_hits(starts):.1%}',
                     onset_F1=round(s['song_macro_f1'], 4), offset_F1=round(s['song_macro_offset_f1'], 4)))
snap_results = pd.DataFrame(rows).set_index('version')
display(snap_results)
print('labelled_notes_right: share of the 4,285 labelled notes whose start is on a right option '
      '(the most possible is 91.1%; "stay" alone is 65.8%).')

Margins chosen per fold (decision tree): {0: 0.15, 1: 0.15, 2: 0.1, 3: 0.1, 4: 0.2}
Margins chosen per fold (random forest): {0: -0.05, 1: 0.0, 2: 0.0, 3: 0.0, 4: 0.05}


,starts_moved,labelled_notes_right,onset_F1,offset_F1
version,,,,
Final chain (no snap),0,65.8%,0.3667,0.2294
Rule: snap to where pitch matches,1411,62.4%,0.3486,0.2191
Decision tree picker,1684,67.7%,0.3781,0.2387
Random forest picker,2388,70.9%,0.3940,0.2507


labelled_notes_right: share of the 4,285 labelled notes whose start is on a right option (the most possible is 91.1%; "stay" alone is 65.8%).


In [4]:
# Step 3: final picker -> validation songs (once) -> export train out-of-fold and val note tables + action histories
sys.path.insert(0, str((HERE / '..').resolve()))
import bp_experiment as bx

# 1. Final model: random forest on ALL labelled training options; margin = best training out-of-fold onset F1
final_margin = max(MARGINS, key=lambda g: f1_on(ALL, choose(rf_prob, g))[0])
final_rf = make_rf().fit(cands.loc[train_rows, CLUES], y[train_rows])
print('Final margin (chosen on training songs only):', final_margin)

# 2. Validation input = the combined chain's validation notes; candidates built the same way (no labels needed)
all_manifest = pd.read_csv(DATA / 'recording_manifest.csv', dtype={'recording_id':str, 'sample_id':str})
val_manifest = all_manifest[all_manifest.split == 'val'].reset_index(drop=True)
assert val_manifest.split.eq('val').all()                                   # never test
val_chain = read(EXPORTS / 'combined_final_val_v1.csv')
cu.validate_notes(val_chain, val_manifest)
val_events, _ = bx.references(val_manifest.to_dict('records'))              # answer key: scoring only
val_refs = pd.DataFrame([dict(note_id=f'{rid}:ref{i:05d}', recording_id=rid, start=e[0], end=e[1], pitch=e[2])
                         for rid, events in val_events.items() for i, e in enumerate(events)])
VCACHE = CACHE_DIR / 'snap_candidates_val_v1.csv'
if VCACHE.exists():
    val_cands = read(VCACHE)
else:
    val_cands = build_candidates(val_chain, val_manifest)                   # ~30 s; no answer key used
    val_cands.to_csv(VCACHE, index=False)
assert set(val_cands.note_id) == set(val_chain.note_id)
val_start = choose(final_rf.predict_proba(val_cands[CLUES])[:, 1], final_margin, val_cands)

def score(table, refs, man):
    out = cu.evaluate_end_to_end(table[cu.NOTE_COLUMNS], refs, man)
    assert out['summary']['recordings'] == len(man)
    return out
val_before = score(val_chain, val_refs, val_manifest)
val_snapped = apply_starts(val_start, val_chain)
val_after = score(val_snapped, val_refs, val_manifest)
before_song = {s['sample_id']: s['f1'] for s in val_before['songs']}
better = sum(s['f1'] > before_song[s['sample_id']] + 1e-12 for s in val_after['songs'])
worse = sum(s['f1'] < before_song[s['sample_id']] - 1e-12 for s in val_after['songs'])
train_snapped = apply_starts(rf_start)                                       # training: out-of-fold picks (step 2)
val_table = pd.DataFrame([
    dict(songs='Training (out-of-fold)', version='Final chain',        notes=len(chain),
         starts_moved=0, onset_F1=snap_results.loc['Final chain (no snap)', 'onset_F1'], offset_F1=snap_results.loc['Final chain (no snap)', 'offset_F1']),
    dict(songs='Training (out-of-fold)', version='Final chain + snap', notes=len(train_snapped),
         starts_moved=len(rf_start), onset_F1=snap_results.loc['Random forest picker', 'onset_F1'], offset_F1=snap_results.loc['Random forest picker', 'offset_F1']),
    dict(songs='Validation (20 songs)', version='Final chain',        notes=len(val_chain), starts_moved=0,
         onset_F1=round(val_before['summary']['song_macro_f1'], 4), offset_F1=round(val_before['summary']['song_macro_offset_f1'], 4)),
    dict(songs='Validation (20 songs)', version='Final chain + snap', notes=len(val_snapped), starts_moved=len(val_start),
         onset_F1=round(val_after['summary']['song_macro_f1'], 4), offset_F1=round(val_after['summary']['song_macro_offset_f1'], 4)),
]).set_index(['songs', 'version'])
display(val_table)
print(f'Validation songs better: {better}, worse: {worse}, unchanged: {len(val_after["songs"]) - better - worse} (onset F1, vs final chain)')
print('Caveat: val was already used to choose the Basic Pitch settings, so it is continued development, not untouched evidence.')

# 3. Export complete note tables + action histories (write once; on a rerun, check the saved files still match)
KINDS = ['is_attack', 'is_louder', 'is_voice_on', 'is_pitch_jump']
def snap_history(table_before, new_start, options, model=None):
    moved = table_before.set_index('note_id').loc[new_start.index]
    opt = options[options.is_stay == 0].set_index(['note_id', 't'])
    kinds = [';'.join(k[3:] for k in KINDS if opt.loc[(n, t), k].max() == 1) for n, t in new_start.items()]
    h = pd.DataFrame({'note_id': new_start.index, 'recording_id': moved.recording_id.to_numpy(), 'action': 'start_snap',
                      'old_start': moved.start.to_numpy(), 'new_start': new_start.to_numpy(), 'moment_kinds': kinds})
    h['shift_ms'] = ((h.new_start - h.old_start) * 1000).round(1)
    h['model'] = model or f'RandomForestClassifier picker, margin {final_margin}; train = out-of-fold by song'
    h['label_version'] = 'answer key (references_train.csv) for labels; provisional'
    order = {n: i for i, n in enumerate(table_before.note_id)}
    return h.assign(_o=h.note_id.map(order)).sort_values('_o').drop(columns='_o').reset_index(drop=True)

def save_once(table, path, man=None):
    if path.exists():
        saved = pd.read_csv(path, dtype={'note_id':str, 'recording_id':str})
        pd.testing.assert_frame_equal(saved, table.reset_index(drop=True), check_dtype=False)
        print('Already saved, unchanged:', path.name)
    elif man is not None:
        cu.export_corrections(table, man, path)
        print('Saved:', path.name)
    else:
        with path.open('x', encoding='utf-8', newline='') as handle:
            table.to_csv(handle, index=False)
        print('Saved:', path.name)

save_once(train_snapped, EXPORTS / 'le_xin_snap_train_oof_v1.csv', manifest)
save_once(snap_history(chain, rf_start, cands), EXPORTS / 'le_xin_snap_train_oof_v1_actions.csv')
save_once(val_snapped, EXPORTS / 'le_xin_snap_val_v1.csv', val_manifest)
save_once(snap_history(val_chain, val_start, val_cands), EXPORTS / 'le_xin_snap_val_v1_actions.csv')

Final margin (chosen on training songs only): 0.0


notes  starts_moved  onset_F1  \
songs                  version                                             
Training (out-of-fold) Final chain          7550             0    0.3667   
                       Final chain + snap   7550          2388    0.3940   
Validation (20 songs)  Final chain          2314             0    0.3904   
                       Final chain + snap   2314           688    0.4205   

                                           offset_F1  
songs                  version                        
Training (out-of-fold) Final chain            0.2294  
                       Final chain + snap     0.2507  
Validation (20 songs)  Final chain            0.2504  
                       Final chain + snap     0.2754

Validation songs better: 18, worse: 1, unchanged: 1 (onset F1, vs final chain)
Caveat: val was already used to choose the Basic Pitch settings, so it is continued development, not untouched evidence.
Already saved, unchanged: le_xin_snap_train_oof_v1.csv
Already saved, unchanged: le_xin_snap_train_oof_v1_actions.csv
Already saved, unchanged: le_xin_snap_val_v1.csv
Already saved, unchanged: le_xin_snap_val_v1_actions.csv


In [5]:
# Step 4: better clues for the picker (v2) -> retrain out-of-fold by song on the training songs and compare with v1
# New clues, all from the audio / Basic Pitch / predicted notes (never the answer key):
#   Basic Pitch : its own onset signal at the note's pitch (and any pitch) at the moment, and its "this pitch is
#                 sounding" signal just before / after (cached by final_pipeline/tools/snap_bp_clues.py, needs basic-pitch)
#   settle      : when does the sung pitch first settle on the note (stays within 0.5 semitone for ~46 ms)?
#                 how far is this moment from that, and how well does the pitch match over the next 100 ms?
#   neighbours  : distance to the previous note's start / end and to the next note's start
#   comparison  : is this moment's attack / loudness rise / pitch jump / Basic Pitch onset the biggest of this
#                 note's options, and how much bigger than at 'stay'?
BP_CLUES     = ['bp_onset_at', 'bp_onset_any', 'bp_note_before', 'bp_note_after', 'bp_note_rise']
SETTLE_CLUES = ['settle_dt', 'settle_abs', 'has_settle', 'match_100', 'steady_100', 'settle_rank', 'match_rank']
NEIGH_CLUES  = ['to_prev_start', 'to_prev_end', 'to_next_start']
COMPARED     = ['attack', 'louder', 'pitch_jump', 'bp_onset_at', 'bp_note_rise']
COMP_CLUES   = [f'{a}_{b}' for a in COMPARED for b in ['vs_stay', 'is_max']]
CLUES_V2 = CLUES + BP_CLUES + SETTLE_CLUES + NEIGH_CLUES + COMP_CLUES

def add_new_clues(options, notes, bp):
    """Same rows as `options`, plus the v2 clue columns."""
    c = options.copy()
    assert len(bp) == len(c) and (bp.note_id.to_numpy() == c.note_id.to_numpy()).all() and np.allclose(bp.t, c.t)
    for col in BP_CLUES[:4]:
        c[col] = bp[col].to_numpy()
    c['bp_note_rise'] = c.bp_note_after - c.bp_note_before           # pitch switches on here?
    # neighbours (previous / next note in the same recording)
    n = notes.sort_values(['recording_id', 'start']).copy()
    g = n.groupby('recording_id')
    n['prev_start'], n['prev_end'], n['next_start'] = g.start.shift(1), g.end.shift(1), g.start.shift(-1)
    n = n.set_index('note_id')
    c['to_prev_start'] = (c.t - c.note_id.map(n.prev_start)).fillna(5.0).clip(upper=5.0)   # 5 s = 'no neighbour'
    c['to_prev_end'] = (c.t - c.note_id.map(n.prev_end)).fillna(5.0).clip(-5.0, 5.0)
    c['to_next_start'] = (c.note_id.map(n.next_start) - c.t).fillna(5.0).clip(upper=5.0)
    # pitch settle + longer pitch match (Hann Qian's cached pyin tracks)
    settle_dt, match_100, steady_100 = pd.Series(np.nan, c.index), pd.Series(FAR, c.index), pd.Series(FAR, c.index)
    for rid, idx in c.groupby('recording_id').groups.items():
        tt, midi = F0[f'{rid}_t'], librosa.hz_to_midi(F0[f'{rid}_f0'])
        settled = {}
        for i in idx:
            nid, t, p = c.at[i, 'note_id'], c.at[i, 't'], c.at[i, 'note_pitch']
            if nid not in settled:
                s0 = n.at[nid, 'start']
                on = np.abs(midi - p) < 0.5                                       # on the note's pitch (no pitch -> False)
                hit = [j for j in np.flatnonzero((tt >= s0 - 0.15) & (tt <= s0 + 0.15)) if on[j:j + 4].size == 4 and on[j:j + 4].all()]
                settled[nid] = tt[hit[0]] if hit else np.nan
            settle_dt.at[i] = t - settled[nid]
            seg = midi[(tt >= t) & (tt < t + 0.10)]
            seg = seg[~np.isnan(seg)]
            if len(seg):
                match_100.at[i] = min(abs(np.median(seg) - p), FAR)
            if len(seg) > 1:
                steady_100.at[i] = min(np.std(seg), FAR)
    c['has_settle'] = settle_dt.notna().astype(float)
    c['settle_abs'] = settle_dt.abs().fillna(1.0)                         # 1 s = 'never settles'
    c['settle_dt'] = settle_dt.fillna(1.0)
    c['match_100'], c['steady_100'] = match_100, steady_100
    # comparison with the note's other options
    g = c.groupby('note_id')
    stay = c[c.is_stay == 1].set_index('note_id')
    for col in COMPARED:
        c[f'{col}_vs_stay'] = c[col] - c.note_id.map(stay[col])
        c[f'{col}_is_max'] = (c[col] == g[col].transform('max')).astype(float)
    c['match_rank'] = g.match_100.rank(method='min')
    c['settle_rank'] = g.settle_abs.rank(method='min')
    return c

CACHE2 = CACHE_DIR / 'snap_candidates_train_v2.csv'
if CACHE2.exists():
    cands2 = read(CACHE2)
    print('Loaded cached v2 candidates:', CACHE2.name)
else:
    cands2 = add_new_clues(cands.drop(columns=['sample_id', 'fold']), chain, read(CACHE_DIR / 'snap_bp_clues_train_v1.csv'))   # ~1 min
    cands2.to_csv(CACHE2, index=False)
    print('Built and cached v2 candidates:', CACHE2.name)
cands2['sample_id'], cands2['fold'] = cands.sample_id.to_numpy(), cands.fold.to_numpy()
assert (cands2.note_id.to_numpy() == cands.note_id.to_numpy()).all() and np.allclose(cands2.t, cands.t)
assert cands2[CLUES_V2].notna().all().all()

rf2_prob, rf2_start, rf2_margin = nested(make_rf, CLUES_V2, cands2)   # ~5-8 minutes
print('Margins chosen per fold (random forest, v2 clues):', rf2_margin)

rows = []
for name, starts in [('Final chain (no snap)', pd.Series(dtype=float)), ('Random forest picker, v1 clues (20)', rf_start),
                     (f'Random forest picker, v2 clues ({len(CLUES_V2)})', rf2_start)]:
    s = cu.evaluate_end_to_end(apply_starts(starts), references, manifest)['summary']
    assert s['recordings'] == len(manifest)
    rows.append(dict(version=name, starts_moved=len(starts), labelled_notes_right=f'{note_hits(starts):.1%}',
                     onset_F1=round(s['song_macro_f1'], 4), offset_F1=round(s['song_macro_offset_f1'], 4)))
snap_results_v2 = pd.DataFrame(rows).set_index('version')
display(snap_results_v2)

# Which clue groups does the v2 forest lean on? (importance summed per group, model on all labelled training options)
imp = pd.Series(make_rf().fit(cands2.loc[train_rows, CLUES_V2], y[train_rows]).feature_importances_, CLUES_V2)
groups = {'v1 clues': CLUES, 'Basic Pitch': BP_CLUES, 'pitch settle / longer match': SETTLE_CLUES,
          'neighbours': NEIGH_CLUES, 'comparison': COMP_CLUES}
display(pd.DataFrame({'clues': {g: len(c) for g, c in groups.items()},
                      'importance': {g: round(imp[c].sum(), 3) for g, c in groups.items()}}))
print('Top 10 clues:', imp.sort_values(ascending=False).head(10).round(3).to_dict())

Loaded cached v2 candidates: snap_candidates_train_v2.csv
Margins chosen per fold (random forest, v2 clues): {0: 0.0, 1: 0.0, 2: 0.0, 3: -0.05, 4: 0.0}


,starts_moved,labelled_notes_right,onset_F1,offset_F1
version,,,,
Final chain (no snap),0,65.8%,0.3667,0.2294
"Random forest picker, v1 clues (20)",2388,70.9%,0.3940,0.2507
"Random forest picker, v2 clues (45)",2546,72.7%,0.4034,0.2568


,clues,importance
v1 clues,20,0.456
Basic Pitch,5,0.159
pitch settle / longer match,7,0.141
neighbours,3,0.102
comparison,10,0.142


Top 10 clues: {'abs_dt': 0.092, 'louder': 0.059, 'pitch_jump': 0.047, 'attack': 0.047, 'settle_abs': 0.045, 'to_prev_end': 0.042, 'steady_100': 0.041, 'bp_note_after': 0.036, 'bp_note_before': 0.036, 'prev_gap': 0.036}


In [6]:
# Step 5: final picker v2 -> validation songs (once) -> export snap v2 (train out-of-fold + val) + action histories
# 1. Final model: random forest with the v2 clues on ALL labelled training options; margin = best training out-of-fold onset F1
final_margin2 = max(MARGINS, key=lambda g: f1_on(ALL, choose(rf2_prob, g))[0])
final_rf2 = make_rf().fit(cands2.loc[train_rows, CLUES_V2], y[train_rows])
print('Final margin, v2 (chosen on training songs only):', final_margin2)

# 2. Validation options = the Step 3 val options + the same new clues (Basic Pitch clues cached by snap_bp_clues.py)
VCACHE2 = CACHE_DIR / 'snap_candidates_val_v2.csv'
if VCACHE2.exists():
    val_cands2 = read(VCACHE2)
else:
    val_cands2 = add_new_clues(val_cands, val_chain, read(CACHE_DIR / 'snap_bp_clues_val_v1.csv'))   # ~20 s; no answer key used
    val_cands2.to_csv(VCACHE2, index=False)
assert (val_cands2.note_id.to_numpy() == val_cands.note_id.to_numpy()).all() and val_cands2[CLUES_V2].notna().all().all()
val_start2 = choose(final_rf2.predict_proba(val_cands2[CLUES_V2])[:, 1], final_margin2, val_cands2)
val_snapped2 = apply_starts(val_start2, val_chain)
val_after2 = score(val_snapped2, val_refs, val_manifest)
v1_song = {s['sample_id']: s['f1'] for s in val_after['songs']}
better2 = sum(s['f1'] > before_song[s['sample_id']] + 1e-12 for s in val_after2['songs'])
worse2 = sum(s['f1'] < before_song[s['sample_id']] - 1e-12 for s in val_after2['songs'])
better_v1 = sum(s['f1'] > v1_song[s['sample_id']] + 1e-12 for s in val_after2['songs'])
worse_v1 = sum(s['f1'] < v1_song[s['sample_id']] - 1e-12 for s in val_after2['songs'])
train_snapped2 = apply_starts(rf2_start)          # training: out-of-fold picks (step 4); their times come from `cands` (Step 1 table)
r = lambda out, k: round(out['summary'][k], 4)
val_table2 = pd.DataFrame([
    dict(songs='Training (out-of-fold)', version='Final chain',           starts_moved=0, **snap_results_v2.iloc[0][['onset_F1', 'offset_F1']]),
    dict(songs='Training (out-of-fold)', version='+ snap v1 (20 clues)',  starts_moved=len(rf_start), **snap_results_v2.iloc[1][['onset_F1', 'offset_F1']]),
    dict(songs='Training (out-of-fold)', version=f'+ snap v2 ({len(CLUES_V2)} clues)', starts_moved=len(rf2_start), **snap_results_v2.iloc[2][['onset_F1', 'offset_F1']]),
    dict(songs='Validation (20 songs)',  version='Final chain',           starts_moved=0, onset_F1=r(val_before, 'song_macro_f1'), offset_F1=r(val_before, 'song_macro_offset_f1')),
    dict(songs='Validation (20 songs)',  version='+ snap v1 (20 clues)',  starts_moved=len(val_start), onset_F1=r(val_after, 'song_macro_f1'), offset_F1=r(val_after, 'song_macro_offset_f1')),
    dict(songs='Validation (20 songs)',  version=f'+ snap v2 ({len(CLUES_V2)} clues)', starts_moved=len(val_start2), onset_F1=r(val_after2, 'song_macro_f1'), offset_F1=r(val_after2, 'song_macro_offset_f1')),
]).set_index(['songs', 'version'])
display(val_table2)
print(f'Validation songs, snap v2 vs final chain: better {better2}, worse {worse2}, unchanged {20 - better2 - worse2}')
print(f'Validation songs, snap v2 vs snap v1:     better {better_v1}, worse {worse_v1}, unchanged {20 - better_v1 - worse_v1}')
print('Caveat: val was already used to choose the Basic Pitch settings, so it is continued development, not untouched evidence.')

# 3. Export complete note tables + action histories (write once; v1 files are left as they are)
MODEL2 = f'RandomForestClassifier picker v2 ({len(CLUES_V2)} clues), margin {final_margin2}; train = out-of-fold by song'
save_once(train_snapped2, EXPORTS / 'le_xin_snap_train_oof_v2.csv', manifest)
save_once(snap_history(chain, rf2_start, cands, MODEL2),EXPORTS / 'le_xin_snap_train_oof_v2_actions.csv')
save_once(val_snapped2, EXPORTS / 'le_xin_snap_val_v2.csv', val_manifest)
save_once(snap_history(val_chain, val_start2, val_cands2, MODEL2), EXPORTS / 'le_xin_snap_val_v2_actions.csv')

Final margin, v2 (chosen on training songs only): 0.0


starts_moved  onset_F1  offset_F1
songs                  version                                                
Training (out-of-fold) Final chain                      0    0.3667     0.2294
                       + snap v1 (20 clues)          2388    0.3940     0.2507
                       + snap v2 (45 clues)          2546    0.4034     0.2568
Validation (20 songs)  Final chain                      0    0.3904     0.2504
                       + snap v1 (20 clues)           688    0.4205     0.2754
                       + snap v2 (45 clues)           708    0.4281     0.2796

Validation songs, snap v2 vs final chain: better 20, worse 0, unchanged 0
Validation songs, snap v2 vs snap v1:     better 14, worse 4, unchanged 2
Caveat: val was already used to choose the Basic Pitch settings, so it is continued development, not untouched evidence.
Saved: le_xin_snap_train_oof_v2.csv
Saved: le_xin_snap_train_oof_v2_actions.csv
Saved: le_xin_snap_val_v2.csv
Saved: le_xin_snap_val_v2_actions.csv
